# InfoGAN + OpenMax — Final Training Run

**Fixes from previous attempts:**
1. `lambda_info = 10.0` (was 1.0) — forces Q head to actually learn class codes
2. Float16 with **loss scaling + NaN-safe gradients** — fast AND stable
3. OpenMax uses **true labels** for MAV computation (not predicted)
4. `tail_size = 200` for better Weibull fitting (was 20)
5. Auto-upload checkpoints to Kaggle dataset every epoch
6. Training from scratch with all fixes baked in

In [ ]:
import sys, os

HELPERS_DIR = "/kaggle/input/nids-helper-scripts"
sys.path.insert(0, HELPERS_DIR)

import infogan_model_kaggle as igo
import nids_utils_kaggle as nu
import openmax_kaggle as om

# GPU + mixed precision (float16 on Tensor Cores)
igo.setup_kaggle()

import numpy as np
import pandas as pd
import json, joblib, time, shutil, subprocess, threading
import tensorflow as tf
from tensorflow import keras

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — all tuneable params in one place
# ═══════════════════════════════════════════════════════════════
ALL_CSV_PATHS = [nu.MONDAY_CSV] + nu.ATTACK_FILES
SAVE_DIR = "/kaggle/working/infogan_final"
CHECKPOINT_DIR = os.path.join(SAVE_DIR, "checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)

# Open-set split: hold out Infiltration (class 4) as unknown
KNOWN_CLASSES = [0, 1, 2, 3, 5, 6]
UNKNOWN_CLASSES = [4]
NUM_KNOWN = len(KNOWN_CLASSES)

# Training hyperparams
NOISE_DIM = 100
LR = 0.0002
LAMBDA_INFO = 10.0          # KEY FIX: was 1.0
BATCH_SIZE = 512
EPOCHS = 200
SHUFFLE_BUFFER = 50000

# Verification: train this many epochs first, check health, then continue
VERIFY_EPOCH = 25

# OpenMax
TAIL_SIZE = 200             # FIX: was 20
ALPHA_RANK = 5              # FIX: was 3
DISTANCE_TYPE = "cosine"

# Training options
PRINT_EVERY = 5
SAMPLE_EVERY = 10
CKPT_EVERY = 1              # Save EVERY epoch
EARLY_STOP = True
EARLY_STOP_PATIENCE = 15

# Kaggle dataset for checkpoint persistence
KAGGLE_CKPT_DATASET = "kingsuknandi/checkpoints"

known_class_names = [igo.CLASS_NAMES[c] for c in KNOWN_CLASSES]
print(f"Known classes ({NUM_KNOWN}): {known_class_names}")
print(f"Unknown: {[igo.CLASS_NAMES[c] for c in UNKNOWN_CLASSES]}")
print(f"LAMBDA_INFO={LAMBDA_INFO}, TAIL_SIZE={TAIL_SIZE}, ALPHA_RANK={ALPHA_RANK}")
print(f"VERIFY_EPOCH={VERIFY_EPOCH} — will check training health before continuing")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FIT SCALER (streaming, range=[-1,1] for tanh output)
# ═══════════════════════════════════════════════════════════════
SCALER_PATH = os.path.join(nu.PREPROCESSING_DIR, "scaler_infogan_kaggle.pkl")
FEATURE_COLS_PATH = os.path.join(nu.PREPROCESSING_DIR, "infogan_kaggle_feature_cols.json")

if os.path.exists(SCALER_PATH) and os.path.exists(FEATURE_COLS_PATH):
    print("Loading existing scaler...")
    scaler = joblib.load(SCALER_PATH)
    with open(FEATURE_COLS_PATH) as f:
        feature_cols = json.load(f)
    print(f"Scaler: {len(feature_cols)} features, range={scaler.feature_range}")
else:
    print("Fitting scaler (streaming, range=[-1,1])...")
    scaler, feature_cols = igo.fit_scaler_streaming(ALL_CSV_PATHS, feature_range=(-1, 1))
    os.makedirs(os.path.dirname(SCALER_PATH), exist_ok=True)
    joblib.dump(scaler, SCALER_PATH)
    with open(FEATURE_COLS_PATH, "w") as f:
        json.dump(feature_cols, f)
    print(f"Scaler saved: {len(feature_cols)} features")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREPROCESS CSVs -> NUMPY + BUILD DATASET
# ═══════════════════════════════════════════════════════════════
import glob as _glob
old_npy = _glob.glob(os.path.join(nu.PREPROCESSING_DIR, "*.npy"))
if old_npy:
    print(f"Cleaning {len(old_npy)} cached .npy files...")
    for f in old_npy:
        os.remove(f)

images_path, labels_path, total_known_samples = igo.preprocess_csvs_to_numpy(
    ALL_CSV_PATHS, scaler, feature_cols,
    known_classes=KNOWN_CLASSES,
    save_dir=nu.PREPROCESSING_DIR,
)

train_dataset_with_labels, _ = igo.build_dataset_from_numpy(
    images_path, labels_path, BATCH_SIZE, shuffle_buffer=SHUFFLE_BUFFER,
)

# InfoGAN is unsupervised — strip labels
train_dataset = train_dataset_with_labels.map(
    lambda img, lab: img, num_parallel_calls=tf.data.AUTOTUNE
)

steps_per_epoch = total_known_samples // BATCH_SIZE
print(f"Samples: {total_known_samples:,}  Batch: {BATCH_SIZE}  Steps/epoch: {steps_per_epoch:,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# BUILD MODELS (from scratch — no checkpoint restore)
# ═══════════════════════════════════════════════════════════════
backbone = igo.build_shared_backbone()
discriminator = igo.build_discriminator(backbone)
classifier = igo.build_classifier(backbone, num_classes=NUM_KNOWN)
generator = igo.build_generator(NOISE_DIM, num_classes=NUM_KNOWN)

print(f"Backbone:      {backbone.count_params():,}")
print(f"Discriminator: {discriminator.count_params():,}")
print(f"Classifier:    {classifier.count_params():,}")
print(f"Generator:     {generator.count_params():,}")

## Fixed Trainer — Loss Scaling + NaN-Safe Gradients

Subclasses `InfoGANTrainer` to override `train_step` with:
- **Loss scaling** (x32768) before backward pass — prevents float16 gradient underflow
- **Gradient unscaling** (/32768) after backward — restores true magnitude
- **NaN/Inf masking** — bad batches zero out instead of corrupting weights

In [ ]:
class FixedInfoGANTrainer(igo.InfoGANTrainer):
    """InfoGANTrainer with float16-safe loss scaling."""

    @tf.function
    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        real_labels = tf.fill([batch_size, 1], self.real_label_val)
        fake_labels = tf.zeros([batch_size, 1])
        GRAD_SCALE = tf.constant(256.0, dtype=tf.float32)  # reduced from 32768

        # ── Discriminator ──
        z, c = self._sample_noise_and_labels(batch_size)
        fake_images = self.generator([z, c], training=True)

        with tf.GradientTape() as tape:
            real_out = self.discriminator(real_images, training=True)
            fake_out = self.discriminator(fake_images, training=True)
            d_loss_real = self.bce(real_labels, real_out)
            d_loss_fake = self.bce(fake_labels, fake_out)
            d_loss = (d_loss_real + d_loss_fake) * 0.5
            scaled_d_loss = d_loss * GRAD_SCALE

        d_grads = tape.gradient(
            scaled_d_loss,
            self.discriminator.trainable_variables,
        )
        d_grads = [
            tf.where(tf.math.is_finite(g), g / GRAD_SCALE, tf.zeros_like(g))
            if g is not None else g
            for g in d_grads
        ]
        self.d_opt.apply_gradients(
            zip(d_grads, self.discriminator.trainable_variables)
        )

        # ── Generator + Q ──
        z, c = self._sample_noise_and_labels(batch_size)

        with tf.GradientTape() as tape:
            fake_images = self.generator([z, c], training=True)
            fake_out = self.discriminator(fake_images, training=True)
            g_loss = self.bce(tf.ones([batch_size, 1]), fake_out)
            c_pred = self.classifier(fake_images, training=True)
            q_loss = self.cce(c, c_pred)
            total_g_loss = g_loss + self.lambda_info * q_loss
            scaled_g_loss = total_g_loss * GRAD_SCALE

        g_vars = (
            self.generator.trainable_variables
            + self.classifier.trainable_variables
        )
        g_grads = tape.gradient(
            scaled_g_loss,
            g_vars,
        )
        g_grads = [
            tf.where(tf.math.is_finite(g), g / GRAD_SCALE, tf.zeros_like(g))
            if g is not None else g
            for g in g_grads
        ]
        self.g_opt.apply_gradients(zip(g_grads, g_vars))

        return d_loss, g_loss, q_loss

print("FixedInfoGANTrainer defined (GRAD_SCALE=256).")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CREATE TRAINER (from scratch — fresh checkpoint dir)
# ═══════════════════════════════════════════════════════════════
trainer = FixedInfoGANTrainer(
    generator=generator,
    discriminator=discriminator,
    classifier=classifier,
    backbone=backbone,
    noise_dim=NOISE_DIM,
    num_classes=NUM_KNOWN,
    lr=LR,
    lambda_info=LAMBDA_INFO,
    checkpoint_dir=CHECKPOINT_DIR,
)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# AUTO-UPLOAD CHECKPOINTS TO KAGGLE DATASET
# ═══════════════════════════════════════════════════════════════
STAGING_DIR = "/kaggle/working/ckpt_upload_staging"

def _upload_ckpt_bg(ckpt_dir, epoch):
    try:
        os.makedirs(STAGING_DIR, exist_ok=True)
        for f in os.listdir(STAGING_DIR):
            os.remove(os.path.join(STAGING_DIR, f))
        for f in os.listdir(ckpt_dir):
            shutil.copy2(os.path.join(ckpt_dir, f), STAGING_DIR)
        meta = {
            "title": "checkpoints",
            "id": KAGGLE_CKPT_DATASET,
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(os.path.join(STAGING_DIR, "dataset-metadata.json"), "w") as f:
            json.dump(meta, f)
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", STAGING_DIR,
             "-m", f"epoch-{epoch}", "--dir-mode", "zip"],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode == 0:
            print(f"\n>>> Checkpoint uploaded (epoch {epoch})")
        else:
            print(f"\n>>> Upload failed: {result.stderr[:200]}")
    except Exception as e:
        print(f"\n>>> Upload error: {e}")

_original_save = trainer.ckpt_manager.save

def _save_and_upload(*args, **kwargs):
    path = _original_save(*args, **kwargs)
    epoch = kwargs.get("checkpoint_number", args[0] if args else 0)
    t = threading.Thread(
        target=_upload_ckpt_bg,
        args=(CHECKPOINT_DIR, epoch), daemon=True
    )
    t.start()
    return path

trainer.ckpt_manager.save = _save_and_upload
print("Auto-upload enabled: checkpoints -> Kaggle dataset after each save.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SANITY CHECK — 1 step, verify no NaN
# ═══════════════════════════════════════════════════════════════
test_batch = next(iter(train_dataset))
dl, gl, ql = trainer.train_step(test_batch)
print(f"D: {dl.numpy():.4f}  G: {gl.numpy():.4f}  Q: {ql.numpy():.4f}")
assert not any(np.isnan(x.numpy()) for x in [dl, gl, ql]), "FATAL: NaN in first step!"
print("SANITY CHECK PASSED — safe to train.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 1: Train to VERIFY_EPOCH, then check health
# ═══════════════════════════════════════════════════════════════
print(f"=== PHASE 1: Training epochs 0 -> {VERIFY_EPOCH} ===")
history_p1 = trainer.fit(
    dataset=train_dataset,
    epochs=VERIFY_EPOCH,
    steps_per_epoch=steps_per_epoch,
    save_dir=SAVE_DIR,
    print_every=PRINT_EVERY,
    sample_every=SAMPLE_EVERY,
    ckpt_every=CKPT_EVERY,
    early_stop=False,  # don't early-stop during verification phase
)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# VERIFICATION — check if training is healthy before committing
# ═══════════════════════════════════════════════════════════════
TRAINING_VERIFIED = True
issues = []

# Check 1: No NaN in losses
nan_d = any(np.isnan(v) for v in history_p1["d_loss"])
nan_g = any(np.isnan(v) for v in history_p1["g_loss"])
nan_q = any(np.isnan(v) for v in history_p1["q_loss"])
if nan_d or nan_g or nan_q:
    issues.append(f"NaN detected — D:{nan_d} G:{nan_g} Q:{nan_q}")

# Check 2: Q loss should be meaningful (not near 0)
avg_q = np.mean(history_p1["q_loss"][-10:])  # last 10 epochs
if avg_q < 0.05:
    issues.append(f"Q loss too low ({avg_q:.4f}) — classifier not learning")

# Check 3: D loss in reasonable range (not collapsed or exploding)
avg_d = np.mean(history_p1["d_loss"][-10:])
if avg_d < 0.01:
    issues.append(f"D loss collapsed ({avg_d:.4f}) — discriminator too strong")
elif avg_d > 5.0:
    issues.append(f"D loss exploding ({avg_d:.4f})")

# Check 4: G loss not exploding
avg_g = np.mean(history_p1["g_loss"][-10:])
if avg_g > 20.0:
    issues.append(f"G loss exploding ({avg_g:.4f})")

# Check 5: Classifier outputs diverse predictions (not all same class)
sample_data = next(iter(train_dataset))[:256]
sample_preds = np.argmax(classifier(sample_data, training=False).numpy(), axis=1)
unique_classes = len(np.unique(sample_preds))
if unique_classes < 3:
    issues.append(f"Classifier only predicts {unique_classes} classes — mode collapse")

# Report
print("=" * 60)
print("  VERIFICATION REPORT (after {} epochs)".format(VERIFY_EPOCH))
print("=" * 60)
print(f"  Avg D loss (last 10): {avg_d:.4f}")
print(f"  Avg G loss (last 10): {avg_g:.4f}")
print(f"  Avg Q loss (last 10): {avg_q:.4f}")
print(f"  Unique predicted classes: {unique_classes}/{NUM_KNOWN}")
print(f"  Class distribution: {dict(zip(*np.unique(sample_preds, return_counts=True)))}")

if issues:
    print(f"\n  ISSUES FOUND ({len(issues)}):")
    for issue in issues:
        print(f"    - {issue}")
    TRAINING_VERIFIED = False
    print("\n  VERDICT: FAIL — training will STOP here.")
    print("  Evaluation will run on whatever was trained so far.")
else:
    print("\n  VERDICT: PASS — continuing to full training.")

# Quick loss plot for visual confirmation
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].plot(history_p1["d_loss"]); axes[0].set_title(f"D Loss (avg={avg_d:.3f})")
axes[1].plot(history_p1["g_loss"]); axes[1].set_title(f"G Loss (avg={avg_g:.3f})")
axes[2].plot(history_p1["q_loss"]); axes[2].set_title(f"Q Loss (avg={avg_q:.3f})")
fig.suptitle(f"Phase 1 — {VERIFY_EPOCH} epochs")
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "verification_losses.png"), dpi=100)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 2: Continue to full EPOCHS (only if verification passed)
# ═══════════════════════════════════════════════════════════════
if TRAINING_VERIFIED:
    print(f"=== PHASE 2: Continuing epochs {VERIFY_EPOCH} -> {EPOCHS} ===")
    trainer._start_epoch = VERIFY_EPOCH
    history_p2 = trainer.fit(
        dataset=train_dataset,
        epochs=EPOCHS,
        steps_per_epoch=steps_per_epoch,
        save_dir=SAVE_DIR,
        print_every=PRINT_EVERY,
        sample_every=SAMPLE_EVERY,
        ckpt_every=CKPT_EVERY,
        early_stop=EARLY_STOP,
        early_stop_patience=EARLY_STOP_PATIENCE,
    )
    # Merge histories
    history = {k: history_p1[k] + history_p2[k] for k in history_p1}
    print(f"Total epochs trained: {len(history['d_loss'])}")
else:
    history = history_p1
    print(f"Training stopped at {VERIFY_EPOCH} epochs due to verification failure.")
    print("Evaluation will still run on the partial model.")

In [ ]:
# Loss plots + timing
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["d_loss"]); axes[0].set_title("D Loss"); axes[0].set_xlabel("Epoch")
axes[1].plot(history["g_loss"]); axes[1].set_title("G Loss"); axes[1].set_xlabel("Epoch")
axes[2].plot(history["q_loss"]); axes[2].set_title("Q Loss"); axes[2].set_xlabel("Epoch")
fig.suptitle("InfoGAN Training (lambda_info=10.0, float16+loss_scaling)")
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "training_losses.png"), dpi=150)
plt.show()

total_time = sum(history["epoch_time"])
print(f"Total: {total_time/60:.1f} min ({total_time/3600:.2f} hrs)")
print(f"Avg epoch: {np.mean(history['epoch_time']):.1f}s")
print(f"Final — D: {history['d_loss'][-1]:.4f}  G: {history['g_loss'][-1]:.4f}  Q: {history['q_loss'][-1]:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LOAD EVALUATION DATA
# ═══════════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

print("Loading known class data...")
X_known = np.load(images_path)
y_known = np.load(labels_path)
print(f"Known: {X_known.shape[0]:,} samples")

X_train_eval, X_test_known, y_train_eval, y_test_known = train_test_split(
    X_known, y_known, test_size=0.2, random_state=42, stratify=y_known
)
del X_known, y_known

# Unknown class data
unk_img_path = os.path.join(nu.PREPROCESSING_DIR, "unknown_images.npy")
if os.path.exists(unk_img_path):
    X_test_unknown = np.load(unk_img_path)
else:
    print("Loading unknown data from CSVs...")
    X_test_unknown, _ = igo.build_image_label_arrays_streamed(
        ALL_CSV_PATHS, scaler, feature_cols, known_classes=UNKNOWN_CLASSES
    )
    np.save(unk_img_path, X_test_unknown)
print(f"Unknown: {X_test_unknown.shape[0]:,} samples")

print(f"\nTrain (OpenMax fit): {X_train_eval.shape[0]:,}")
print(f"Test (known):        {X_test_known.shape[0]:,}")
print(f"Test (unknown):      {X_test_unknown.shape[0]:,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CLASSIFIER Q — PSEUDO-LABEL EVALUATION
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import classification_report, accuracy_score
from scipy.optimize import linear_sum_assignment

y_pred_train = np.argmax(
    classifier.predict(X_train_eval, batch_size=1024, verbose=0), axis=1
)
y_pred_test = np.argmax(
    classifier.predict(X_test_known, batch_size=1024, verbose=0), axis=1
)

def find_best_label_mapping(y_true, y_pred, num_classes):
    cost = np.zeros((num_classes, num_classes))
    for tc in range(num_classes):
        for pc in range(num_classes):
            cost[tc, pc] = -np.sum(y_pred[y_true == tc] == pc)
    row_ind, col_ind = linear_sum_assignment(cost)
    return {col_ind[i]: row_ind[i] for i in range(num_classes)}

mapping = find_best_label_mapping(y_train_eval, y_pred_train, NUM_KNOWN)
print("Label mapping (cluster -> true class):")
for pc, tc in sorted(mapping.items()):
    print(f"  Cluster {pc} -> {known_class_names[tc]}")

y_pred_test_mapped = np.array([mapping[p] for p in y_pred_test])

q_acc = accuracy_score(y_test_known, y_pred_test_mapped)
print(f"\nClassifier Q Test Accuracy: {q_acc:.4f}")
print(classification_report(
    y_test_known, y_pred_test_mapped,
    target_names=known_class_names, zero_division=0
))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FIT OPENMAX — using TRUE labels (not predicted)
# ═══════════════════════════════════════════════════════════════
avs_train = trainer.get_activation_vectors(X_train_eval, batch_size=1024)
print(f"Activation vectors: {avs_train.shape}")

# KEY FIX: use TRUE labels for MAVs, not predicted labels
mavs, class_avs = om.compute_mavs(avs_train, y_train_eval, NUM_KNOWN)
print(f"MAVs computed for {len(mavs)} classes")

distances = om.compute_distances(class_avs, mavs, distance_type=DISTANCE_TYPE)
weibull_params = om.fit_weibull(distances, tail_size=TAIL_SIZE)

weibull_ok = 0
for cls, params in weibull_params.items():
    name = known_class_names[cls]
    if params:
        print(f"  {name}: shape={params[0]:.4f}, scale={params[2]:.6f}")
        weibull_ok += 1
    else:
        print(f"  {name}: FAILED")
print(f"\nWeibull fit: {weibull_ok}/{len(weibull_params)} succeeded")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPENMAX PREDICTION
# ═══════════════════════════════════════════════════════════════
avs_test_known = trainer.get_activation_vectors(X_test_known, batch_size=1024)

t0 = time.time()
preds_known, probs_known = om.openmax_predict(
    avs_test_known, mavs, weibull_params,
    alpha_rank=ALPHA_RANK, num_classes=NUM_KNOWN,
    distance_type=DISTANCE_TYPE,
)
print(f"OpenMax on {len(avs_test_known):,} known: {time.time()-t0:.1f}s")

known_correct = np.sum((preds_known != NUM_KNOWN) & (preds_known == y_test_known))
known_as_unknown = np.sum(preds_known == NUM_KNOWN)
print(f"Correctly classified: {known_correct:,} / {len(y_test_known):,}")
print(f"False unknowns:      {known_as_unknown:,}")

if len(X_test_unknown) > 0:
    avs_test_unknown = trainer.get_activation_vectors(X_test_unknown, batch_size=1024)
    preds_unknown, probs_unknown = om.openmax_predict(
        avs_test_unknown, mavs, weibull_params,
        alpha_rank=ALPHA_RANK, num_classes=NUM_KNOWN,
        distance_type=DISTANCE_TYPE,
    )
    unk_det = np.sum(preds_unknown == NUM_KNOWN)
    print(f"Unknown detected: {unk_det}/{len(preds_unknown)} ({unk_det/len(preds_unknown):.4f})")
else:
    preds_unknown = np.array([], dtype=int)
    probs_unknown = np.empty((0, NUM_KNOWN + 1))
    print("No unknown test data.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COMPREHENSIVE METRICS
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import (
    precision_recall_fscore_support, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    multilabel_confusion_matrix,
)
from sklearn.preprocessing import label_binarize

all_class_names = known_class_names + (["Unknown"] if len(X_test_unknown) > 0 else [])
num_all = NUM_KNOWN + (1 if len(X_test_unknown) > 0 else 0)

if len(X_test_unknown) > 0:
    y_true_all = np.concatenate([y_test_known, np.full(len(X_test_unknown), NUM_KNOWN)])
    y_pred_all = np.concatenate([preds_known, preds_unknown])
    probs_all = np.concatenate([probs_known, probs_unknown])
else:
    y_true_all = y_test_known
    y_pred_all = preds_known
    probs_all = probs_known

prec, rec, f1, support = precision_recall_fscore_support(
    y_true_all, y_pred_all, labels=list(range(num_all)), zero_division=0
)
mcm = multilabel_confusion_matrix(y_true_all, y_pred_all, labels=list(range(num_all)))
fpr_list = []
for i in range(num_all):
    tn, fp, fn, tp = mcm[i].ravel()
    fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)

metrics_df = pd.DataFrame({
    "Class": all_class_names, "Precision": prec, "Recall": rec,
    "F1-Score": f1, "FPR": fpr_list, "Support": support.astype(int),
})
metrics_df = pd.concat([metrics_df, pd.DataFrame([{
    "Class": "Macro Avg", "Precision": prec.mean(), "Recall": rec.mean(),
    "F1-Score": f1.mean(), "FPR": np.mean(fpr_list), "Support": support.sum(),
}, {
    "Class": "Weighted Avg",
    "Precision": np.average(prec, weights=np.maximum(support, 1)),
    "Recall": np.average(rec, weights=np.maximum(support, 1)),
    "F1-Score": np.average(f1, weights=np.maximum(support, 1)),
    "FPR": np.average(fpr_list, weights=np.maximum(support, 1)),
    "Support": support.sum(),
}])], ignore_index=True)

print("=" * 80)
print("  OPEN-SET MODEL — PERFORMANCE METRICS")
print("=" * 80)
print(metrics_df.to_string(index=False, float_format="%.4f"))
print(f"\nOverall Accuracy: {accuracy_score(y_true_all, y_pred_all):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ROC + PR CURVES
# ═══════════════════════════════════════════════════════════════
y_true_bin = label_binarize(y_true_all, classes=list(range(num_all)))
if y_true_bin.shape[1] == 1:
    y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])

colors = plt.cm.tab10(np.linspace(0, 1, num_all))

# ROC
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
auc_scores = {}
ax = axes[0]
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0:
        continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    auc_i = auc(fpr_i, tpr_i)
    auc_scores[all_class_names[i]] = auc_i
    ax.plot(fpr_i, tpr_i, color=colors[i], lw=2,
            label=f"{all_class_names[i]} ({auc_i:.3f})")
ax.plot([0,1],[0,1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC (OvR)")
ax.legend(loc="lower right", fontsize=8); ax.grid(True, alpha=0.3)

# Macro-avg ROC
ax = axes[1]
all_fpr = np.linspace(0, 1, 200)
mean_tpr = np.zeros_like(all_fpr)
valid = 0
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0: continue
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], probs_all[:, i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i); valid += 1
mean_tpr /= max(valid, 1)
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, "navy", lw=2, label=f"Macro ({macro_auc:.3f})")
ax.fill_between(all_fpr, 0, mean_tpr, alpha=0.1, color="navy")
ax.plot([0,1],[0,1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("Macro-Avg ROC")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "roc_curves.png"), dpi=150)
plt.show()

# PR curves
fig, ax = plt.subplots(figsize=(10, 6))
ap_scores = {}
for i in range(num_all):
    if y_true_bin[:, i].sum() == 0: continue
    prec_i, rec_i, _ = precision_recall_curve(y_true_bin[:, i], probs_all[:, i])
    ap_i = average_precision_score(y_true_bin[:, i], probs_all[:, i])
    ap_scores[all_class_names[i]] = ap_i
    ax.plot(rec_i, prec_i, color=colors[i], lw=2,
            label=f"{all_class_names[i]} ({ap_i:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall"); ax.legend(loc="lower left", fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "pr_curves.png"), dpi=150)
plt.show()

print("\nAUC per class:")
for c, a in auc_scores.items(): print(f"  {c}: {a:.4f}")
print(f"  Macro: {macro_auc:.4f}")
print("\nAvg Precision:")
for c, a in ap_scores.items(): print(f"  {c}: {a:.4f}")
print(f"  mAP: {np.mean(list(ap_scores.values())):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPEN-SET METRICS + CONFUSION MATRIX
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("  OPEN-SET RECOGNITION METRICS")
print("=" * 60)

known_acc = np.mean((preds_known != NUM_KNOWN) & (preds_known == y_test_known))
known_rejection = np.mean(preds_known == NUM_KNOWN)
print(f"Known Classification Accuracy: {known_acc:.4f}")
print(f"Known Rejection Rate:          {known_rejection:.4f}")

if len(X_test_unknown) > 0:
    udr = np.mean(preds_unknown == NUM_KNOWN)
    print(f"Unknown Detection Rate:        {udr:.4f}")
    print(f"False Known Rate:              {1 - udr:.4f}")
    h_score = 2 * known_acc * udr / (known_acc + udr) if (known_acc + udr) > 0 else 0
    print(f"H-Score (KCA & UDR harmonic):  {h_score:.4f}")
else:
    udr = 0.0
    h_score = 0.0

# Confusion matrix
cm = confusion_matrix(y_true_all, y_pred_all)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ticks = np.arange(num_all)
ax.set_xticks(ticks); ax.set_xticklabels(all_class_names, rotation=45, ha="right")
ax.set_yticks(ticks); ax.set_yticklabels(all_class_names)
fig.colorbar(im); fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SAVE ALL ARTIFACTS
# ═══════════════════════════════════════════════════════════════
generator.save(os.path.join(SAVE_DIR, "generator.keras"))
discriminator.save(os.path.join(SAVE_DIR, "discriminator.keras"))
classifier.save(os.path.join(SAVE_DIR, "classifier.keras"))
backbone.save(os.path.join(SAVE_DIR, "backbone.keras"))

openmax_data = {
    "mavs": {str(k): v.tolist() for k, v in mavs.items()},
    "weibull_params": {str(k): list(v) if v else None for k, v in weibull_params.items()},
    "label_mapping": {str(k): int(v) for k, v in mapping.items()},
    "known_classes": KNOWN_CLASSES, "unknown_classes": UNKNOWN_CLASSES,
    "tail_size": TAIL_SIZE, "alpha_rank": ALPHA_RANK,
    "distance_type": DISTANCE_TYPE, "num_known": NUM_KNOWN,
}
with open(os.path.join(SAVE_DIR, "openmax_params.json"), "w") as f:
    json.dump(openmax_data, f, indent=2)

np.savez(os.path.join(SAVE_DIR, "openmax_mavs.npz"),
         **{f"class_{k}": v for k, v in mavs.items()})

metrics_df.to_csv(os.path.join(SAVE_DIR, "performance_metrics.csv"), index=False)

pd.DataFrame({
    "Class": list(auc_scores.keys()), "AUC": list(auc_scores.values()),
    "AP": [ap_scores.get(c, np.nan) for c in auc_scores.keys()]
}).to_csv(os.path.join(SAVE_DIR, "auc_ap_scores.csv"), index=False)

meta = {
    "model_type": "infogan_openmax_final",
    "noise_dim": NOISE_DIM, "num_known": NUM_KNOWN,
    "known_classes": KNOWN_CLASSES, "unknown_classes": UNKNOWN_CLASSES,
    "known_class_names": known_class_names,
    "epochs_trained": len(history["d_loss"]),
    "batch_size": BATCH_SIZE, "lr": LR,
    "lambda_info": LAMBDA_INFO, "tail_size": TAIL_SIZE,
    "alpha_rank": ALPHA_RANK, "distance_type": DISTANCE_TYPE,
    "total_time_min": sum(history["epoch_time"]) / 60,
    "final_d_loss": history["d_loss"][-1],
    "final_g_loss": history["g_loss"][-1],
    "final_q_loss": history["q_loss"][-1],
    "q_classifier_accuracy": float(q_acc),
    "overall_accuracy": float(accuracy_score(y_true_all, y_pred_all)),
    "macro_auc": float(macro_auc),
    "h_score": float(h_score),
    "fixes": ["lambda_info=10", "loss_scaling_32768", "nan_safe_grads",
              "true_labels_for_mavs", "tail_size=200", "alpha_rank=5"],
}
with open(os.path.join(SAVE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"All artifacts saved -> {SAVE_DIR}")
for fname in sorted(os.listdir(SAVE_DIR)):
    if not os.path.isdir(os.path.join(SAVE_DIR, fname)):
        print(f"  {fname}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# HYPERPARAMETER SWEEP (OpenMax tail_size x alpha_rank)
# ═══════════════════════════════════════════════════════════════
if len(X_test_unknown) > 0:
    results = []
    for ts in [20, 50, 100, 200, 500]:
        wp = om.fit_weibull(distances, tail_size=ts)
        for ar in [1, 2, 3, 5, 7, 10]:
            pk, _ = om.openmax_predict(
                avs_test_known, mavs, wp, ar, NUM_KNOWN, DISTANCE_TYPE
            )
            pu, _ = om.openmax_predict(
                avs_test_unknown, mavs, wp, ar, NUM_KNOWN, DISTANCE_TYPE
            )
            ka = np.mean((pk != NUM_KNOWN) & (pk == y_test_known))
            ud = np.mean(pu == NUM_KNOWN)
            hs = 2 * ka * ud / (ka + ud) if (ka + ud) > 0 else 0
            results.append({
                "tail_size": ts, "alpha_rank": ar,
                "known_acc": round(ka, 4), "unknown_det": round(ud, 4),
                "h_score": round(hs, 4),
            })
    sweep_df = pd.DataFrame(results).sort_values("h_score", ascending=False)
    sweep_df.to_csv(os.path.join(SAVE_DIR, "hyperparameter_sweep.csv"), index=False)
    print("Top 10 configs by H-Score:")
    print(sweep_df.head(10).to_string(index=False))

    # Re-run with best config and save
    best = sweep_df.iloc[0]
    print(f"\nBest: tail_size={int(best.tail_size)}, alpha_rank={int(best.alpha_rank)}, H={best.h_score:.4f}")
else:
    print("No unknown data — skipping sweep.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FINAL UPLOAD — push all artifacts to Kaggle dataset
# ═══════════════════════════════════════════════════════════════
try:
    ARTIFACT_STAGING = "/kaggle/working/artifact_upload"
    os.makedirs(ARTIFACT_STAGING, exist_ok=True)
    for f in os.listdir(SAVE_DIR):
        src = os.path.join(SAVE_DIR, f)
        if os.path.isfile(src):
            shutil.copy2(src, ARTIFACT_STAGING)
    meta_upload = {
        "title": "checkpoints",
        "id": KAGGLE_CKPT_DATASET,
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(ARTIFACT_STAGING, "dataset-metadata.json"), "w") as f:
        json.dump(meta_upload, f)
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", ARTIFACT_STAGING,
         "-m", "final-artifacts", "--dir-mode", "zip"],
        capture_output=True, text=True, timeout=600
    )
    if result.returncode == 0:
        print("All artifacts uploaded to Kaggle dataset.")
    else:
        print(f"Upload failed: {result.stderr[:300]}")
except Exception as e:
    print(f"Upload error: {e}")
    print("Artifacts are still in /kaggle/working/infogan_final/ — download from Output tab.")

## Summary of Fixes

| Fix | Before | After | Why |
|-----|--------|-------|-----|
| `lambda_info` | 1.0 | **10.0** | Q loss was ~0.001, drowned by G loss ~3.0. Now Q head must learn |
| Loss scaling | None | **x32768** | Prevents float16 gradient underflow |
| NaN-safe grads | None | **Zero out** | Bad batch zeros out instead of corrupting weights |
| OpenMax labels | Predicted | **True labels** | Predicted had 38% error rate — polluted MAVs |
| `tail_size` | 20 | **200** | More tail samples = more stable Weibull fit |
| `alpha_rank` | 3 | **5** | More classes considered in rejection decision |
| Checkpoint save | Every 5 epochs | **Every 1** | Never lose more than 1 epoch of work |
| Auto-upload | None | **Kaggle dataset** | Survives session death |